In [ ]:
import re
import string
import numpy as np
import pandas as pd
import csv
import json
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
verbs = ["agree", "believe", "bet", "consider", "decide", "expect", "feel",
         "figure", "find", "guess", "hear", "hope", "imagine", "know", "mean",
         "notice", "read", "realize", "remember", "say", "see", "show",
         "suppose", "take", "teach", "tell", "thank", "think", "understand",
         "wish", "worry"]
additional_verbs = ["accept", "acknowledge", "add", "affirm", "allege",
                    "announce", "answer", "anticipate", "argue", "assert",
                    "attest", "boast", "brag", "calculate", "caution",
                    "certify", "claim", "comment", "complain", "conceal",
                    "concede", "confess", "confirm", "declare", "deduce",
                    "demonstrate", "deny", "determine", "deny", "determine",
                    "disclose", "doubt", "dream", "emphasize", "establish",
                    "estimate", "explain", "fear", "forget", "gloat", "growl",
                    "guarantee", "hate", "hint", "holler", "hoot", "hypothesize",
                    "ignore", "imply", "indicate","infer", "insist", "intimate",
                    "joke", "learn", "like", "maintain", "mention", "moan",
                    "mumble", "murmur", "muse", "mutter", "note", "observe",
                    "opine", "perceive", "plead", "predict", "presume", "pretend",
                    "proclaim", "promise", "propose", "prove", "reason", "recall",
                    "reckon", "recognize", "recollect", "regret", "reiterate",
                    "remark", "repeat", "reply", "report", "request", "resent",
                    "respond", "reveal", "scream", "sense", "shout", "shriek",
                    "signal", "signify", "speculate", "stammer", "state",
                    "suggest", "suspect", "swear", "testify", "theorize", "trust",
                    "verify", "vow", "wail", "warn", "whine", "whisper", "wonder",
                    "write", "yell"]
all_verbs = verbs + additional_verbs
all_verbs_set = set(all_verbs)
len(all_verbs)

In [ ]:
# constituency parsing
!pip install spacy
!python -m spacy download en_core_web_md
# !python -m spacy download en_core_web_trf
# !pip install -U spacy[cuda12x]
# !python -m spacy download en_core_web_sm

In [ ]:
import spacy

# spacy.prefer_gpu()
spacy.require_cpu() # default to using cpu

nlp = spacy.load("en_core_web_md")
# nlp = spacy.load("en_core_web_trf")
# nlp = spacy.load("en_core_web_sm")

def get_span(left_id, right_id, doc):
  return doc[left_id:right_id+1]

def get_subj_type(subj):
  if subj.lemma_.lower() == "i":
    return "I"
  elif subj.lemma_.lower() == "it":
    return "It"
  elif subj.pos_ == "PRON":
    return "pronoun"
  else:
    return "NP"

# get the object of the verb
def overt_subject(verb):
  for child in verb.children:
    if child.dep_.startswith("nsubj") or child.dep_.startswith("csubj") or child.dep_ == "expl":
      return child
  return None

def extract_subject(verb):
  """
  Get the subject of the verb by:
    1) checking overt subject on verb
    2) going through control/raising chains (xcomp, aux) or the coordinated verb
        to find a governing verb/AUX that has a subject.
  Return: subject
  """
  # 1) finding the overt subject on the verb
  subject = overt_subject(verb)
  if subject:
    return subject

  # 2) finding the governing verb/AUX with a subject
  #    Typical chains: VERB <-xcomp- VERB, VERB <-aux/-auxpass- AUX
  current_word = verb
  visited = set()
  while current_word.head is not current_word and current_word.i not in visited:
    visited.add(current_word.i)
    # only find if the verb is in a control/raising relation or is coordinated with another verb
    if current_word.dep_ in {"xcomp","aux","auxpass"} or (current_word.dep_ == "conj" and current_word.head.pos_ in {"VERB","AUX"}):
      parent = current_word.head
      if parent.pos_ in {"VERB","AUX"}:
        subject = overt_subject(parent)
        if subject:
          return subject
          current_word = parent
          continue
    break

  return None

def get_embedded_clause(embedded_verb):
  """
  Check the type of complementizer and get the span (left edge id and right edge id) of the embedded clause
  Return:
    comp_type: "that" for overt 'that' complementizer, "omitted" for omitted complementizer, "other" for other complementizer types
    left_i: the left edge of the embedded clause (including the complementizer if there is one), None if it's other complementizer
    right_i: the right edge of the embedded clause (including the complementizer if there is one), None if it's other complementizer
  """
  left_i = embedded_verb.left_edge.i
  right_i = embedded_verb.right_edge.i

  that_complementizer = None
  other_complementizer = None

  for ch in embedded_verb.children:
    # print("printing the label of each embedded verb's child:", ch.dep_)
    # exclude other types of complementizer
    if ch.dep_ == "mark":
      if ch.lemma_.lower() == "that":
        if that_complementizer is None or ch.i < that_complementizer.i:
          that_complementizer = ch
      else:
        other_complementizer = ch
  # the subtree of the embedded verb does not include the complementizer
  # so the leftmost boundary should be the complementizer
  if that_complementizer:
    left_i = min(that_complementizer.i, left_i)
    return "that", left_i, right_i
  # if other complementizer is used
  if other_complementizer:
    return "other", None, None
  # if the complementizer is omitted
  return "omitted", left_i, right_i


def extract_ccomps(doc):
  """
  return one record per (matrix verb, ccomp head) where the complementizer is 'that' or omitted.
  Ignores xcomp entirely (not returned). Still robust to coordination and non-root predicates.
  """
  rows = []
  for sent in doc.sents:
    for tok in sent:
      if tok.pos_ in {"VERB","AUX"} and tok.lemma_.lower() in all_verbs_set: # tok is the matrix predicate
        embedded_clause = False
        comp_type = "none" # default to no embedded clause

        # extract the subject
        matrix_subj = extract_subject(tok)
        matrix_subj_left_i = matrix_subj.left_edge.i if matrix_subj else None
        matrix_subj_right_i = matrix_subj.right_edge.i if matrix_subj else None
        matrix_subj_span = get_span(matrix_subj_left_i, matrix_subj_right_i, matrix_subj.doc) if matrix_subj else None

        # matrix_span_verb = get_span(tok.left_edge.i,tok.i,tok.doc) if tok.i != tok.left_edge.i else get_span(0,tok.i,tok.doc)

        matrix_span_no_verb = get_span(tok.left_edge.i,tok.i-1,tok.doc) if tok.i != tok.left_edge.i else get_span(0,tok.i-1,tok.doc)
        # in case of control/raising: need to find the subj as the start of the matrix clause
        if matrix_subj and str(matrix_subj.text) not in str(matrix_span_no_verb):
          matrix_span_no_verb = get_span(matrix_subj.left_edge.i,tok.i-1,tok.doc)

        matrix_span_verb = str(matrix_span_no_verb) + " " + tok.text

        # check if the verb has a ccomp
        for ch in tok.children:
          # print("printing the label of each main verb's child:", ch.dep_)
          # the child is the embedded verb, connected to the matrix verb by ccomp (i.e., matrix verb -ccomp-> embedded verb)

          if ch.dep_ == "ccomp":
            comp_type, embedded_clause_left_i, embedded_clause_right_i = get_embedded_clause(ch)
            if comp_type == "other":
              continue  # skip other complementizers for now

            embedded_clause = True
            embedded_subj = extract_subject(ch)
            embedded_clause_span = get_span(embedded_clause_left_i, embedded_clause_right_i, ch.doc)
            # if the embedded clause starts with wh-words, i.e., taking a wh-constituent, skip for now
            if str(embedded_clause_span).split()[0].lower() in ["which", "how", "what", "why", "when", "where"] :
              comp_type = "other"
              continue
            embedded_clause_omit_that_left_i = embedded_clause_left_i + 1 if comp_type == "that" else embedded_clause_left_i # left edge of the comp, not including "that" if there is one
            # embedded_clause_omit_that_span = get_span(embedded_clause_omit_that_left_i, embedded_clause_right_i, ch.doc)

            # use the right bound of embedded subj if there is an embedded subj, else use the position of the embedded verb (not including the verb, henche the ch.i-1)
            # cc_onset = embedded_subj.right_edge.i - embedded_clause_left_i if embedded_subj else ch.i - embedded_clause_left_i
            # cc_onset = ch.i - embedded_clause_left_i # onset of the embedded clause, including the matrix subj
            cc_onset_span = get_span(embedded_clause_omit_that_left_i, embedded_subj.right_edge.i, ch.doc) if embedded_subj else get_span(embedded_clause_omit_that_left_i, ch.i-1, ch.doc)
            embedded_clause_one_word = get_span(embedded_clause_omit_that_left_i, embedded_clause_omit_that_left_i, ch.doc) if embedded_clause_right_i - embedded_clause_left_i  >= 1 else embedded_clause_span

            print(tok.i == tok.left_edge.i)

            rows.append({
                # sentence
                "sentence": sent.text,

                # matrix predicate
                "matrix_predicate": tok.text,
                "matrix_predicate_lemma": tok.lemma_,
                "matrix_predicate_id": tok.i, # absolute position of the verb in the sent
                # "matrix_predicate_position": tok.i - matrix_subj_right_i if matrix_subj else None, # relative position of the verb in the matrix sentence (i.e. # of words after the matrix subject)
                "matrix_predicate_position": tok.i - tok.left_edge.i, # relative position of the verb in the matrix sentence

                # matrix subject
                "matrix_subject_head": matrix_subj.text if matrix_subj else None,
                "matrix_subject_head_id": matrix_subj.i if matrix_subj else None,
                "matrix_subject_span": str(matrix_subj_span) if matrix_subj else None,
                "matrix_subject_type": get_subj_type(matrix_subj) if matrix_subj else None,

                # matrix clause
                "matrix_span_no_verb": str(matrix_span_no_verb),
                "matrix_span_verb": str(matrix_span_verb),

                # complementizer
                "complementizer": comp_type,
                "complement_type": "ccomp",
                "matrix_predicate_to_cc": embedded_clause_left_i - tok.i - 1,

                # embedded clause
                # "cc_onset": cc_onset - 1 if embedded_subj and comp_type == "that" else cc_onset, # diff (i.e. cc_onset-1) when the comp is "that"
                # use the right bound of embedded subj if there is an embedded subj, else use the position of the embedded verb, should be the same as len(cc_onset_span)
                "cc_onset": embedded_subj.right_edge.i - embedded_clause_omit_that_left_i + 1 if embedded_subj else ch.i - embedded_clause_omit_that_left_i,
                # "cc_onset": cc_onset - 1 if comp_type == "that" else cc_onset,
                "cc_onset_span": str(cc_onset_span) if cc_onset_span else None,
                "cc_reminder": embedded_clause_right_i - embedded_subj.right_edge.i if embedded_subj else embedded_clause_right_i - ch.i + 1, # need to include the
                "embedded_clause_span": str(embedded_clause_span) if embedded_clause_span else None,
                "embedded_subject_head": embedded_subj.text if embedded_subj else None,
                "embedded_subject_head_id": embedded_subj.i if embedded_subj else None,
                "embedded_subject_span": str(get_span(embedded_subj.left_edge.i, embedded_subj.right_edge.i, embedded_subj.doc)) if embedded_subj else None,

                # "whole_matrix_one_word_omit_that": str(get_span(0,embedded_clause_left_i-1, tok.doc)) + " " + str(embedded_clause_one_word),

                "onset_omit_that_span": str(get_span(tok.left_edge.i,embedded_clause_left_i-1, tok.doc)) + " " + str(cc_onset_span) if cc_onset_span else None,
                "one_word_omit_that": str(get_span(tok.left_edge.i,embedded_clause_left_i-1, tok.doc)) + " " + str(embedded_clause_one_word) if tok.i != tok.left_edge.i else str(get_span(0,embedded_clause_left_i-1, tok.doc)) + " " + str(embedded_clause_one_word),
                "one_word_with_that": str(get_span(tok.left_edge.i,embedded_clause_left_i-1, tok.doc)) + " that " + str(embedded_clause_one_word) if tok.i != tok.left_edge.i else str(get_span(0,embedded_clause_left_i-1, tok.doc)) + " that " + str(embedded_clause_one_word)

                })

        # if there is no child with ccomp (i.e., no embedded clause) or if it is "whether" or other complementizer
        if not embedded_clause:
          rows.append({
              # sentence
              "sentence": sent.text,

              # matrix predicate
              "matrix_predicate": tok.text,
              "matrix_predicate_lemma": tok.lemma_,
              "matrix_predicate_id": tok.i, # absolute position of the verb in the sent
              "matrix_predicate_position": tok.i - matrix_subj_right_i if matrix_subj else None, # relative position of the verb in the matrix sentence (i.e. # of words after the matrix subject)

              # matrix subject
              "matrix_subject_head": matrix_subj.text if matrix_subj else None,
              "matrix_subject_head_id": matrix_subj.i if matrix_subj else None,
              "matrix_subject_span": str(matrix_subj_span) if matrix_subj else None,
              "matrix_subject_type": get_subj_type(matrix_subj) if matrix_subj else None,

              # matrix clause
              "matrix_span_no_verb": str(matrix_span_no_verb),
              "matrix_span_verb": str(matrix_span_verb),

              # complementizer and embedded clause
              "complementizer": comp_type,
              "complement_type": None,
              "matrix_predicate_to_cc": None,

              "cc_onset": None,
              "cc_onset_span": None,
              "cc_reminder": None,
              "embedded_clause_span": None,
              "embedded_subject_head": None,
              "embedded_subject_head_id": None,
              "embedded_subject_span": None,

              "onset_omit_that_span": None,
              "one_word_omit_that": None,
              "one_word_with_that": None
          })
  return rows

In [ ]:
import json

if __name__ == "__main__":
  ex1 = "You need to understand that when you send an SAR, the information that you receive will be very helpful."
  ex2 = "She must have felt the only way she could get back at Cindy is through Caylee."
  ex3 = "i sent her a long ass text explaining that i didn't feel the relationship was what i needed right now but i cared about her a lot."
  ex4 = "and the narrator said there's nothing to move the plot..."
  ex5 = "I don't know whether I should go to the party."
  ex6 = "Among those with infections such as tick-borne encephalitis, influenza, Lyme, mycoplasma, and bartonella (as examples) researchers have constantly noted the healthier the immune system, the less likely one is to be infected and, if infected, the less severe the course of the disease."
  ex7 = "Muslims assert that Moe only fought defensively, that the verses only apply \"in time of war\"."
  ex8 = "Decided another indistinct trail leading to the control looked pretty good"
  # examples = [ex1, ex2, ex3, ex4, ex5, ex6, ex7]
  examples = [ex1, ex2, ex3, ex7, ex8]
  # examples = declarative_ds_that_test["sentence"]
  results = []
  for ex in examples:
    parse = nlp(ex)
    result = extract_ccomps(parse)
    # results.extend(result)
    result_print = json.dumps(result, indent=4)
    print(result_print)

In [ ]:
# downgrade to 3.6.0 to run dolma dataset
# see: https://github.com/huggingface/datasets/issues/7693
# !pip install datasets
!pip install datasets==3.6.0

In [ ]:
from datasets import load_dataset
# 13,095,416 sentences in total
ds = load_dataset("allenai/dolma","v1_6-sample",split="train")

In [ ]:
# remove sentences from The Stack (source=="stack-dedup") since most are code-related
# this results in 12,462,749 sentences (originally: 13,095,416)
def filter_out_source(batch, source):
  return [t != source for t in batch["source"]]

ds_filter = ds.filter(lambda batch: filter_out_source(batch, "stack-dedup"), batched=True, batch_size=10000)

In [ ]:
ds_filter_shuffle = ds_filter.shuffle(seed=1024)
ds_filter_shuffle = ds_filter_shuffle.add_column("doc_id", list(range(len(ds_filter_shuffle))))

In [ ]:
# ds_filter_shuffle.to_csv("/content/drive/MyDrive/comp_drop/dolma_v1_6-sample.csv")
ds_sentences_1 = ds_filter_shuffle.select(range(500000))
ds_sentences_2 = ds_filter_shuffle.select(range(500000,1000000))
ds_sentences_3 = ds_filter_shuffle.select(range(1000000,1500000))
ds_sentences_4 = ds_filter_shuffle.select(range(1500000,2000000))
# ds_sentences_1 = ds_shuffle.take(1000000) # if using streaming when loading the dataset

In [ ]:
from datasets import load_dataset
ds_sentences_1 = load_dataset("csv", data_files="/content/drive/MyDrive/comp_drop/dolma_v1_6-sample_1.csv")["train"]

In [ ]:
_split_re = re.compile(r'(?:\n|\s{2,}|(?<=[.!?])\s+)')

def clean_sent(sent):
  # replace special symbols
  sent = sent.replace("’", "'")
  sent = sent.replace("“", "\"")
  sent = sent.replace("”", "\"")
  sent = sent.replace("…", "...")
  sent = sent.replace("–", "-")
  # remove leading bullet-like patterns such as "*.", "-", etc.
  return re.sub(r'^\s*[\*\-]+\s*\.?\s*', '', sent).strip()

def split_clean_sent(batch):
  texts = batch["text"]
  doc_ids = batch["doc_id"]
  sources = batch["source"]

  sentence = []
  doc_id = []
  sent_id = []
  sent_source = []

  for id, source, text in zip(doc_ids, sources, texts):
    if not text:
      continue
    sentences = _split_re.split(str(text))
    sentences = [clean_sent(sent) for sent in sentences if clean_sent(sent)]

    for i, sent in enumerate(sentences):
      sentence.append(sent)
      doc_id.append(id)
      sent_id.append(i)
      sent_source.append(source)

  return {"sent_id_in_doc": sent_id, "sentence": sentence, "doc_id": doc_id, "source": sent_source}

In [ ]:
# 5019354 sentences
ds_sentences_1_split = ds_sentences_1.map(
    split_clean_sent,
    batched=True,
    remove_columns=ds_sentences_1.column_names,
    batch_size=128,
    num_proc=4,
    )

In [ ]:
# 12,441,615 sentences in ds_sentences_1_split, load 1,000,000 to ds_sentences_test
# ds_sentences_1_split.to_csv("/content/drive/MyDrive/comp_drop/dolma_v1_6-sample_1_split.csv")
# from datasets import Dataset
# ds_sentences_test = Dataset.from_dict(ds_sentences_1_split[:1000000])

from datasets import load_dataset

ds_sentences_1_split = load_dataset("csv", data_files="/content/drive/MyDrive/comp_drop/dolma_v1_6-sample_1_split.csv")["train"]
ds_sentences_test = ds_sentences_1_split.select(range(1000000))

In [ ]:
def process_batch(batch):
  sentences = batch["sentence"]
  doc_ids = batch["doc_id"]
  sent_ids = batch["sent_id_in_doc"]
  sources = batch["source"]

  # process the sentences
  all_rows = []
  for doc, sent_id, doc_id, source in zip(nlp.pipe(sentences), sent_ids, doc_ids, sources):
    for sent in doc.sents:
      sent_doc = sent.as_doc() # the doc of each sentence to avoid index problems
      results = extract_ccomps(sent_doc)
      if not results:
        continue
      for r in results:
        if not isinstance(r, dict):
          raise TypeError(f"Expected dict, got {type(r)}")
        r.setdefault("sentence", sent.text.strip())
        r["doc_id"] = doc_id
        r["sent_id_in_doc"] = sent_id
        r["source"] = source
      all_rows.extend(results)

  if not all_rows:
    return {"sentence": [], "sent_id_in_doc":[], "doc_id":[], "source":[]}

  # add the keys
  all_keys = set()
  for r in all_rows:
    all_keys.update(r.keys())

  # put "sentence" as the first column and reorder the rest
  ordered_keys = (["sentence"] if "sentence" in all_keys else []) + sorted(k for k in all_keys if k != "sentence")
  result = {k: [r.get(k) for r in all_rows] for k in ordered_keys}

  return result

In [ ]:
# 1,000,000 sentences took 14min
if __name__ == "__main__":
  ds_processed_test = ds_sentences_test.map(
    process_batch,
    batched=True,
    remove_columns=ds_sentences_test.column_names,
    batch_size=128,
    num_proc=4,
    )

In [ ]:
ds_processed_test.to_csv("/content/drive/MyDrive/comp_drop/dolma_v1_6-sample_test_processed.csv")